In [0]:
USE vendas_pecas.camada_bronze;

In [0]:
CREATE OR REFRESH STREAMING TABLE tb_arquivos_bronze
AS SELECT *, 
_metadata.file_path as file_path,
_metadata.file_modification_time as ingest_date
FROM STREAM read_files(
  "/Volumes/vendas_pecas/default/my/processar/",
  format => "csv",
  delimiter => ",",
  header => "true",
  inferColumnTypes => "false",
  schemaEvolutionMode => "rescue"
);

In [0]:
-- USE vendas_pecas.camada_bronze;
-- --DEIXAR INCREMENTAL - MERGE INTO
-- CREATE OR REPLACE TABLE tb_bronze
-- AS
-- SELECT *
-- FROM tb_arquivos_bronze

In [0]:
MERGE WITH SCHEMA EVOLUTION INTO 
 tb_bronze tb
USING 
  (
    SELECT *
    FROM (
      SELECT *, 
        ROW_NUMBER() OVER(
            PARTITION BY id_nf
            ORDER BY ingest_date DESC
        ) AS row_num
      FROM tb_arquivos_bronze
    )
    WHERE 
      row_num = 1
  ) AS source_table 
ON 
  tb.id_nf = source_table.id_nf
WHEN 
  MATCHED
  AND source_table.ingest_date > tb.ingest_date 
THEN 
  UPDATE SET *
WHEN
 NOT MATCHED 
THEN
  INSERT *